# Notebook 6: Chain-of-Thought → Tree-of-Thought → RLM

The evolution of LLM reasoning:

| Technique | Structure | Who decides decomposition? |
|-----------|-----------|---------------------------|
| **Chain-of-Thought** | Linear chain | The prompt template |
| **Tree-of-Thought** | Branching tree | A search algorithm |
| **RLM** | Programmatic tree | The model itself (via code) |

Each step gives the model more autonomy over how it reasons.

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")

from rlm_core import LLMClient, RLMEngine, Sandbox
from rlm_core.visualizer import tree_to_text
import json

class SimulatedClient:
    def __init__(self):
        self.call_count = 0
        self.total_prompt_tokens = 0
        self.total_completion_tokens = 0
    
    def completion(self, prompt, **kwargs):
        self.call_count += 1
        self.total_prompt_tokens += len(prompt.split())
        self.total_completion_tokens += 15
        
        class Result:
            def __init__(self, text, pt, ct):
                self.text = text
                self.prompt_tokens = pt
                self.completion_tokens = ct
        
        # Respond based on context
        if "step by step" in prompt.lower():
            text = "Step 1: Read the context. Step 2: The answer is in the middle. Step 3: The secret code is BLUE-FALCON-42. Final answer: BLUE-FALCON-42"
        elif "evaluate" in prompt.lower() or "score" in prompt.lower():
            text = "Score: 8/10. This reasoning path correctly identifies the answer."
        elif "secret" in prompt.lower() or "code" in prompt.lower():
            text = '''import re
match = re.search(r'secret code is ([A-Z0-9-]+)', context)
FINAL(match.group(1) if match else "Not found")'''
        else:
            text = "The answer based on the context is: information found."
        
        return Result(text, len(prompt.split()), 15)

client = SimulatedClient()

with open("../data/samples/needle_haystack.txt") as f:
    document = f.read()
print(f"Loaded document: {len(document)} chars")

## Chain-of-Thought (CoT)

**The idea:** Add "Let's think step by step" to the prompt. The model then reasons through the problem linearly before giving a final answer.

**Published:** Wei et al., 2022 — one of the most influential papers in modern AI.

**Structure:** A → B → C → D → Answer (a linear chain)

In [ ]:
def chain_of_thought(client, question, context):
    """Solve a question using Chain-of-Thought prompting."""
    prompt = f"""Context: {context[:1500]}

Question: {question}

Let's think step by step:"""
    
    result = client.completion(prompt, temperature=0.0)
    return {
        "answer": result.text,
        "tokens": result.prompt_tokens + result.completion_tokens,
        "method": "Chain-of-Thought",
    }

# Run CoT
cot_result = chain_of_thought(client, "What is the secret code?", document)
print("=== Chain-of-Thought ===")
print(f"Answer: {cot_result['answer']}")
print(f"Tokens: {cot_result['tokens']}")
print(f"\nNotice: CoT is LINEAR — one chain of reasoning, no branching.")

## Tree-of-Thought (ToT)

**The idea:** Instead of one reasoning chain, generate MULTIPLE chains and evaluate which is best. Like a chess player considering multiple moves.

**Published:** Yao et al., 2023

**Structure:**
```
        Question
       /   |   \
    Path1 Path2 Path3
      |     |     |
   Score  Score  Score
      \    |    /
      Best Path → Answer
```

In [ ]:
def tree_of_thought(client, question, context, n_paths=3):
    """Solve a question using Tree-of-Thought reasoning."""
    paths = []
    
    # Generate multiple reasoning paths
    for i in range(n_paths):
        prompt = f"""Context: {context[:1500]}

Question: {question}

Reasoning path {i+1} of {n_paths}. Think about this from a different angle:"""
        
        result = client.completion(prompt, temperature=0.8)
        paths.append(result.text)
    
    # Evaluate each path
    eval_prompt = f"""Question: {question}

Here are {n_paths} reasoning paths:
{chr(10).join(f'Path {i+1}: {p[:200]}' for i, p in enumerate(paths))}

Evaluate each path. Which is most correct? Score each 1-10:"""
    
    eval_result = client.completion(eval_prompt, temperature=0.0)
    
    total_tokens = sum(len(p.split()) for p in paths) + len(eval_result.text.split())
    
    return {
        "paths": paths,
        "evaluation": eval_result.text,
        "answer": paths[0],  # In practice, pick the highest-scored
        "tokens": total_tokens,
        "method": "Tree-of-Thought",
        "n_paths": n_paths,
    }

# Run ToT
tot_result = tree_of_thought(client, "What is the secret code?", document)
print("=== Tree-of-Thought ===")
print(f"Generated {tot_result['n_paths']} reasoning paths")
for i, path in enumerate(tot_result['paths']):
    print(f"  Path {i+1}: {path[:80]}...")
print(f"Evaluation: {tot_result['evaluation'][:100]}...")
print(f"Tokens: {tot_result['tokens']}")
print(f"\nNotice: ToT explores BREADTH — multiple paths, then selects the best.")

## RLM — The Model Programs Its Own Reasoning

**The idea:** Don't constrain the model to a chain or tree. Give it a code sandbox and let it decide how to decompose the problem. It might use regex, chunking, recursion, or any combination.

**The key difference:** CoT and ToT structure reasoning from the OUTSIDE (via prompts). RLMs let the model structure reasoning from the INSIDE (via code).

In [ ]:
engine = RLMEngine(client=client, max_depth=3)
rlm_result = engine.run("What is the secret code?", document)

print("=== RLM (Recursive Language Model) ===")
print(f"Answer: {rlm_result.answer}")
print(f"LLM calls: {rlm_result.total_llm_calls}")
print(f"Max depth: {rlm_result.max_depth_reached}")
print(f"\nRecursion tree:")
print(tree_to_text(rlm_result.root_node))
print(f"\nNotice: RLM writes CODE to decide its own reasoning strategy.")

## Side-by-Side Comparison

In [ ]:
import matplotlib.pyplot as plt

print("=" * 60)
print(f"{'Method':<20} {'Answer':<30} {'Tokens':<10}")
print("=" * 60)
print(f"{'CoT':<20} {str(cot_result['answer'])[:30]:<30} {cot_result['tokens']:<10}")
print(f"{'ToT (3 paths)':<20} {str(tot_result['answer'])[:30]:<30} {tot_result['tokens']:<10}")
print(f"{'RLM':<20} {str(rlm_result.answer)[:30]:<30} {rlm_result.total_llm_calls:<10} calls")
print("=" * 60)

# Visual comparison
fig, ax = plt.subplots(figsize=(10, 4))

methods = ['Chain-of-Thought\n(Linear)', 'Tree-of-Thought\n(Branching)', 'RLM\n(Programmatic)']
# Conceptual complexity score
flexibility = [1, 3, 5]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

bars = ax.bar(methods, flexibility, color=colors, width=0.6)
ax.set_ylabel('Model Autonomy')
ax.set_title('Evolution of LLM Reasoning: From Constrained to Free')

for bar, label in zip(bars, ['Prompt decides\nstructure', 'Search decides\nstructure', 'Model decides\nstructure']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
            label, ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## The Key Insight

| | CoT | ToT | RLM |
|---|---|---|---|
| **Reasoning structure** | Linear chain | Fixed-width tree | Arbitrary (model decides) |
| **Who designs the structure?** | Prompt engineer | Search algorithm | The model via code |
| **Branching** | None | Fixed (n paths) | Dynamic (model decides when to branch) |
| **Can adapt strategy?** | No | Limited | Yes — can peek, grep, chunk, recurse |
| **Context handling** | Stuffed in prompt | Stuffed in prompt | External variable (unlimited) |

The progression: **constrained → structured → free**.

CoT says "think step by step." ToT says "consider multiple paths." RLM says "write whatever code you need."

## Exercise

Pick a reasoning problem (e.g., "How many countries border France?") and solve it with all three approaches. Compare the reasoning traces.

In [ ]:
your_question = "How many countries border France?"
your_context = """France is bordered by Belgium, Luxembourg, Germany, Switzerland, 
Italy, Spain, Andorra, and Monaco. It also borders the English Channel, 
the Atlantic Ocean, and the Mediterranean Sea."""

print("=== Your CoT ===")
print(chain_of_thought(client, your_question, your_context)["answer"][:200])

print("\n=== Your ToT ===")
print(tree_of_thought(client, your_question, your_context)["answer"][:200])

print("\n=== Your RLM ===")
r = engine.run(your_question, your_context)
print(f"Answer: {r.answer}")
print(tree_to_text(r.root_node))

## Key Takeaways

1. **CoT** adds "think step by step" — simple, effective for straightforward reasoning
2. **ToT** generates multiple paths and evaluates them — better for problems with multiple solution strategies
3. **RLM** lets the model write code to decompose problems — maximum flexibility, handles unbounded context
4. The progression is about giving models **more autonomy** over their reasoning process
5. RLMs are not always better — CoT is simpler and cheaper for easy tasks. Use the right tool for the job.

## What's Next?

Notebook 7 explores **why** RLMs are needed: the fundamental problem of long context (attention is O(n^2), KV cache grows linearly) and how quantization helps.